# QuantAccelerator workbench · FINRA daily short-sale volume

**Understand → Trust → Research.** This notebook walks one unfamiliar dataset from raw files to a point-in-time research
table, with you deciding at every gate:

1. **Understand** — the Data Research Agent reads the documentation, profiles every table, and studies the data the way a
   quant researcher would (coverage, missing data, distributions, update dynamics, structural breaks). Every number
   comes from a logged tool run; every figure is shown as it is made.
2. **Data Brief** — what the data is, what the agent did, what it found, what it can and cannot support.
3. **Gate G0** — you approve, edit or reject the proposed cleaning. Raw data is never modified.
4. **Trust** — the PIT agent works out when each record became public and tradable. **Gate G1**: you sign the rule.
5. **Research view** — one canonical point-in-time table; snapshots only ever see what was knowable.
6. **Audit** — check a research script against the signed contract.

It needs a model server reachable at `LLM_BASE_URL` (any OpenAI-compatible endpoint; see the README).

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                                    # paths are relative to the project root
os.environ["QA_DATASET"] = "finra"                     # one dataset per kernel, set before the import
os.environ.setdefault("LLM_BASE_URL", "http://127.0.0.1:8000/v1")      # any OpenAI-compatible endpoint
os.environ.setdefault("LLM_MODEL", "Qwen/Qwen2.5-72B-Instruct-AWQ")
from quantaccelerator.notebook import Session
s = Session()

## 1 · Understand
Pass 1 (semantics) reads the docs and profiles the tables; pass 2 (profiling) investigates the data. Each step prints one line saying what the agent is examining; the results and figures are in the Data Brief below. It takes about 10–15 minutes. (`Session(verbose=True)` also shows each result and figure as it is made.)

In [ ]:
card = s.understand()

## 2 · The Data Brief

In [ ]:
s.brief()

Ask follow-up questions. The agent answers with tools and cites them; it says so when the data cannot answer.

In [ ]:
s.ask("Is TotalVolume something narrower?")

## 3 · Gate G0: cleaning
Each proposed step is a card: **red** removes rows, **amber** flags rows (they stay, with a new column), **blue** adds a missing-value indicator. Each card shows the problem, the exact action, how many rows it touches, the evidence runs and example rows. Nothing is applied until you approve.

In [ ]:
s.review_cleaning()

In [ ]:
s.approve_cleaning(drop_steps=[], note="reviewed")   # or s.reject_cleaning("why")

## 4 · Trust: when was each record knowable?
The PIT agent proposes an availability rule with its evidence, alternatives and open questions, and a plain-words line says when each record would first be used. **You decide:** approve it, or correct it. The audit below measures every pipeline against the rule you sign, so a wrong rule here makes a leaky pipeline look clean.

In [ ]:
s.time()

In [ ]:
# Gate G1: read the rule and its plain-words line above, then run ONE of these:
# s.approve("why the rule is right")
# s.correct({"source_field": "shvol.Date", "tradable_at": "next_trading_day_open"}, "why")

## 5 · The point-in-time research view

In [ ]:
view = s.pit_view()
view.head()

In [ ]:
s.snapshot(["2024-03-28", "2024-04-01"], ["AAPL", "TSLA", "NVDA"])   # 1 Apr sees Thursday's file (Good Friday)

## 6 · Audit a research script against the signed contract

In [ ]:
s.audit("notebooks/finra_signal_leak_sameday.py")

In [ ]:
s.export_html()